# Tutorial 15: Migrating Entries Between Pools

`laila.forget` is **pool-local** — it only removes a blob from one pool. To move entries from one backend to another you read, write to the destination, then forget on the source.

You will:

- Set up a `warm` pool and a `cold` pool
- Populate `warm` with several entries
- Loop: `remember` from `warm` with `persist=False`, `memorize` to `cold`, `forget` from `warm`
- Verify the source is empty and the destination holds everything

**Prerequisites:** `pip install "laila-core[hdf5]"`.

In [ ]:
import numpy as np

import laila
from laila.pool import FilesystemPool, HDF5Pool

warm = FilesystemPool(nickname="warm")
cold = HDF5Pool(nickname="cold")
laila.memory.extend(warm, pool_nickname="warm")
laila.memory.extend(cold, pool_nickname="cold")

print("warm gid:", warm.global_id)
print("cold gid:", cold.global_id)

## Populate the source

We seed five entries in `warm`. Their `global_id` values are what the migration loop will iterate over.

In [ ]:
entries = [
    laila.constant(data=np.random.randn(4, 4), nickname=f"mig_{i}")
    for i in range(5)
]
for e in entries:
    laila.memorize(e, pool_nickname="warm").wait()

gids = [e.global_id for e in entries]
print("seeded:", len(gids), "entries in warm")

## Migration loop

Three steps per entry:

1. `remember(gid, pool_nickname="warm", persist=False)` — fetch without re-caching into the alpha pool.
2. `memorize(entry, pool_nickname="cold")` — write to the destination.
3. `forget(gid, pool_nickname="warm")` — drop the source copy.

The `persist=False` is important. With the default `persist=True` the read would also cache into the alpha pool, defeating the purpose of the move.

In [ ]:
for gid in gids:
    entry = laila.remember(gid, pool_nickname="warm", persist=False).wait()
    if isinstance(entry, list):
        entry = entry[0]
    laila.memorize(entry, pool_nickname="cold").wait()
    laila.forget(gid, pool_nickname="warm").wait()

print("migration complete")

## Verify

Direct pool introspection: `warm._keys()` should be empty, and `cold._keys()` should contain all five gids.

In [ ]:
warm_keys = list(warm._keys())
cold_keys = list(cold._keys())
print("warm keys remaining:", len(warm_keys))
print("cold keys present:  ", len(cold_keys))
print("all original gids in cold:", all(g in cold_keys for g in gids))

## Recover from cold

`remember(..., pool_nickname="cold")` now resolves every entry through the new pool. The data is byte-identical to the source — pool migration preserves the entry's `global_id` and payload exactly.

In [ ]:
recovered = laila.remember(gids[0], pool_nickname="cold", persist=False).wait()
if isinstance(recovered, list):
    recovered = recovered[0]
print("first recovered shape:", recovered.data.shape)

## Sharp edges

| Concern | Mitigation |
|---|---|
| `forget` only touches one pool | Iterate every pool that may hold a copy. |
| `persist=True` would re-cache during migration | Pass `persist=False` on the read step. |
| Concurrent producers writing to `warm` mid-migration | Lock or pause writes outside LAILA — there is no built-in lease. |

## Summary

- Migration = read with `persist=False`, write to destination, forget on source.
- `forget` is pool-local: wipe every pool you want clean.
- Gids and payloads survive intact, so callers using nicknames see no break.